## 1. Khai báo thư viện và thiết lập đường dẫn project

Ở bước đầu tiên, notebook cần khai báo các thư viện cần thiết và xác định đúng đường dẫn làm việc của project. Việc thiết lập đường dẫn rõ ràng giúp các cell phía sau đọc dữ liệu từ đúng thư mục `data/processed` và lưu kết quả kiểm tra vào thư mục `results`.

Trong project này, dữ liệu đầu vào của giai đoạn cuối kỳ không lấy trực tiếp từ dữ liệu Kaggle gốc, mà được kế thừa từ kết quả xử lý ở giữa kỳ. Vì vậy, thư mục quan trọng nhất ở bước này là `data/processed`.

In [13]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

cwd = Path.cwd()

if cwd.name.lower() == "notebooks":
    PROJECT_DIR = cwd.parent
else:
    PROJECT_DIR = cwd

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = PROJECT_DIR / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

PROJECT_DIR: d:\HK2_NAM3\KTDL_KPTT\CUOI_KY
PROCESSED_DIR: d:\HK2_NAM3\KTDL_KPTT\CUOI_KY\data\processed
RESULTS_DIR: d:\HK2_NAM3\KTDL_KPTT\CUOI_KY\results


## 2. Kiểm tra sự tồn tại của các file dữ liệu đã xử lý

Bước này kiểm tra xem các file dữ liệu đã xử lý từ giữa kỳ có tồn tại đầy đủ trong thư mục `data/processed` hay không. Đây là bước quan trọng trước khi chạy thuật toán, vì nếu thiếu một trong các file đầu vào thì quá trình khai thác tập phổ biến và sinh luật kết hợp có thể bị lỗi hoặc thiếu thông tin phân tích.

Các file được kiểm tra bao gồm ma trận giao dịch, dữ liệu giỏ hàng, bảng ánh xạ sản phẩm và các bảng dữ liệu sạch phục vụ phân tích mở rộng.

In [14]:
required_processed_files = [
    "basket_data_sample.csv",
    "basket_matrix_sparse.pkl",
    "product_mapping_sample.csv",
    "order_product_pairs_sample.csv",
    "instacart_subset_for_rules.csv",
    "clean_aisles.csv",
    "clean_departments.csv",
    "clean_order_products_prior.csv",
    "clean_orders_prior.csv",
    "clean_products.csv"
]

file_check_rows = []

for file_name in required_processed_files:
    file_path = PROCESSED_DIR / file_name
    exists = file_path.exists()
    size_mb = file_path.stat().st_size / (1024 * 1024) if exists else 0

    file_check_rows.append({
        "file_name": file_name,
        "exists": exists,
        "size_mb": round(size_mb, 2)
    })

file_check_df = pd.DataFrame(file_check_rows)
display(file_check_df)

missing_files = file_check_df[file_check_df["exists"] == False]["file_name"].tolist()

if len(missing_files) == 0:
    print("Trạng thái: Tất cả file processed cần thiết đều tồn tại.")
else:
    print("Trạng thái: Thiếu file processed.")
    print(missing_files)

,file_name,exists,size_mb
0,basket_data_sample.csv,True,4.80
1,basket_matrix_sparse.pkl,True,3.35
2,product_mapping_sample.csv,True,0.17
3,order_product_pairs_sample.csv,True,21.38
4,instacart_subset_for_rules.csv,True,52.73
5,clean_aisles.csv,True,0.00
6,clean_departments.csv,True,0.00
7,clean_order_products_prior.csv,True,581.73
8,clean_orders_prior.csv,True,95.12
9,clean_products.csv,True,2.11


Trạng thái: Tất cả file processed cần thiết đều tồn tại.


In [ ]:
## 3. Đọc các file dữ liệu chính cho giai đoạn cuối kỳ

Sau khi xác nhận các file đã tồn tại, notebook tiến hành đọc các file dữ liệu chính. Các file dạng `.csv` được đọc bằng `pd.read_csv()`, còn file `.pkl` được đọc bằng `pd.read_pickle()` vì đây là file nhị phân lưu object của Python.

Ý nghĩa của các file chính như sau:

| File | Vai trò |
| --- | --- |
| `basket_matrix_sparse.pkl` | Ma trận giao dịch - sản phẩm, là đầu vào chính cho Apriori và FP-Growth |
| `product_mapping_sample.csv` | Ánh xạ `product_id` sang tên sản phẩm, quầy hàng và nhóm ngành hàng |
| `basket_data_sample.csv` | Dữ liệu giỏ hàng đã gom theo từng đơn hàng |
| `order_product_pairs_sample.csv` | Dữ liệu dạng từng dòng là một cặp `order_id` và `product_id` |
| `instacart_subset_for_rules.csv` | Dữ liệu tổng hợp dùng cho phân tích mở rộng theo sản phẩm, nhóm hàng, thời gian và hành vi mua lại |

Trong đó, `basket_matrix_sparse.pkl` là file quan trọng nhất cho bước khai thác tập phổ biến.

In [15]:
basket_data = pd.read_csv(PROCESSED_DIR / "basket_data_sample.csv")
product_mapping = pd.read_csv(PROCESSED_DIR / "product_mapping_sample.csv")
order_product_pairs = pd.read_csv(PROCESSED_DIR / "order_product_pairs_sample.csv")
instacart_subset = pd.read_csv(PROCESSED_DIR / "instacart_subset_for_rules.csv")

basket_matrix = pd.read_pickle(PROCESSED_DIR / "basket_matrix_sparse.pkl")

print("Trạng thái: Load dữ liệu processed thành công.")
print("Kiểu dữ liệu basket_matrix:", type(basket_matrix))

Trạng thái: Load dữ liệu processed thành công.
Kiểu dữ liệu basket_matrix: <class 'pandas.DataFrame'>


## 4. Kiểm tra kích thước các bảng dữ liệu

Cell này tổng hợp số dòng và số cột của các bảng dữ liệu chính. Mục đích là xác nhận rằng dữ liệu đầu vào của giai đoạn cuối kỳ khớp với kết quả đã tạo ra ở giữa kỳ.

Các thông tin cần quan tâm gồm số lượng giao dịch, số lượng sản phẩm, số dòng dữ liệu chi tiết đơn hàng và số dòng dữ liệu tổng hợp phục vụ phân tích. Nếu các kích thước này đúng với kỳ vọng, có thể xem dữ liệu đã được copy và đọc đúng.

In [16]:
shape_summary = pd.DataFrame([
    {
        "dataset": "basket_data_sample.csv",
        "rows": basket_data.shape[0],
        "columns": basket_data.shape[1],
        "meaning": "Mỗi dòng là một giao dịch đã gom theo order_id"
    },
    {
        "dataset": "basket_matrix_sparse.pkl",
        "rows": basket_matrix.shape[0],
        "columns": basket_matrix.shape[1],
        "meaning": "Ma trận giao dịch - sản phẩm dùng cho thuật toán"
    },
    {
        "dataset": "product_mapping_sample.csv",
        "rows": product_mapping.shape[0],
        "columns": product_mapping.shape[1],
        "meaning": "Mapping product_id sang product_name, aisle, department"
    },
    {
        "dataset": "order_product_pairs_sample.csv",
        "rows": order_product_pairs.shape[0],
        "columns": order_product_pairs.shape[1],
        "meaning": "Mỗi dòng là một sản phẩm trong một đơn hàng"
    },
    {
        "dataset": "instacart_subset_for_rules.csv",
        "rows": instacart_subset.shape[0],
        "columns": instacart_subset.shape[1],
        "meaning": "Dữ liệu tổng hợp để phân tích mở rộng"
    }
])

display(shape_summary)

,dataset,rows,columns,meaning
0,basket_data_sample.csv,75519,2,Mỗi dòng là một giao dịch đã gom theo order_id
1,basket_matrix_sparse.pkl,75519,3000,Ma trận giao dịch - sản phẩm dùng cho thuật toán
2,product_mapping_sample.csv,3000,4,"Mapping product_id sang product_name, aisle, d..."
3,order_product_pairs_sample.csv,579185,3,Mỗi dòng là một sản phẩm trong một đơn hàng
4,instacart_subset_for_rules.csv,579185,16,Dữ liệu tổng hợp để phân tích mở rộng


## 5. Kiểm tra các cột dữ liệu quan trọng

Sau khi kiểm tra kích thước dữ liệu, bước tiếp theo là xem danh sách cột của từng bảng. Việc này giúp xác định các thuộc tính cần thiết cho những bước xử lý sau có đầy đủ hay không.

Một số cột quan trọng gồm:

| Cột | Ý nghĩa |
| --- | --- |
| `product_id` | Mã định danh sản phẩm, được dùng trong thuật toán |
| `product_name` | Tên sản phẩm, dùng để hiển thị trong báo cáo và ứng dụng demo |
| `aisle_name` | Tên quầy hàng, dùng cho phân tích mở rộng theo quầy |
| `department_name` | Nhóm ngành hàng, dùng cho phân tích theo nhóm sản phẩm |
| `reordered` | Cho biết sản phẩm có được mua lại hay không |
| `order_dow` | Ngày trong tuần khi đơn hàng được đặt |
| `order_hour_of_day` | Giờ trong ngày khi đơn hàng được đặt |

Các cột này không chỉ phục vụ thuật toán mà còn giúp phân tích kết quả theo góc nhìn kinh doanh.

In [17]:
print("basket_data columns:")
print(basket_data.columns.tolist())

print("\nproduct_mapping columns:")
print(product_mapping.columns.tolist())

print("\norder_product_pairs columns:")
print(order_product_pairs.columns.tolist())

print("\ninstacart_subset columns:")
print(instacart_subset.columns.tolist())

basket_data columns:
['order_id', 'danh_sach_product_id']

product_mapping columns:
['product_id', 'product_name', 'aisle_name', 'department_name']

order_product_pairs columns:
['order_id', 'product_id', 'product_name']

instacart_subset columns:
['order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order', 'days_since_prior_order_filled', 'is_first_order', 'product_id', 'product_name', 'aisle_id', 'aisle_name', 'department_id', 'department_name', 'add_to_cart_order', 'reordered']


## 6. Kiểm tra ma trận giao dịch - sản phẩm

`basket_matrix` là cấu trúc dữ liệu trung tâm của bài toán khai thác luật kết hợp. Trong ma trận này, mỗi dòng tương ứng với một đơn hàng, mỗi cột tương ứng với một sản phẩm, và giá trị trong ô cho biết sản phẩm có xuất hiện trong đơn hàng hay không.

Cách biểu diễn này phù hợp với yêu cầu đầu vào của các thuật toán Apriori và FP-Growth, vì hai thuật toán này cần dữ liệu giao dịch ở dạng có hoặc không có sản phẩm.

Ví dụ:

| Giá trị | Ý nghĩa |
| --- | --- |
| `1` | Sản phẩm xuất hiện trong đơn hàng |
| `0` | Sản phẩm không xuất hiện trong đơn hàng |

In [18]:
print("Kích thước basket_matrix:", basket_matrix.shape)
display(basket_matrix.head())

Kích thước basket_matrix: (75519, 3000)


product_id,10,34,45,49,79,95,116,117,130,141,...,49519,49520,49533,49583,49585,49605,49610,49621,49628,49683
order_id,,,,,,,,,,,,,,,,,,,,,
28,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
67,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
71,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
109,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
122,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 7. Kiểm tra định dạng nhị phân của ma trận

Các thuật toán khai thác tập phổ biến yêu cầu dữ liệu đầu vào ở dạng nhị phân hoặc boolean. Vì vậy, cần kiểm tra một mẫu dữ liệu trong `basket_matrix` để đảm bảo ma trận chỉ chứa hai giá trị `0` và `1`.

Nếu ma trận chỉ chứa `0` và `1`, dữ liệu có thể được chuyển sang dạng boolean để sử dụng cho FP-Growth và Apriori. Nếu xuất hiện giá trị khác, cần kiểm tra lại bước tạo ma trận giao dịch.

In [19]:
sample_matrix = basket_matrix.iloc[:1000, :100]
unique_values = pd.unique(sample_matrix.astype("int8").values.ravel())

print("Các giá trị xuất hiện trong mẫu kiểm tra:")
print(unique_values)

valid_binary = set(unique_values).issubset({0, 1})

if valid_binary:
    print("Trạng thái: Ma trận hợp lệ, mẫu kiểm tra chỉ chứa 0 và 1.")
else:
    print("Trạng thái: Ma trận có giá trị khác 0 và 1, cần kiểm tra lại.")

Các giá trị xuất hiện trong mẫu kiểm tra:
[0 1]
Trạng thái: Ma trận hợp lệ, mẫu kiểm tra chỉ chứa 0 và 1.


## 8. Kiểm tra ánh xạ giữa product_id và thông tin sản phẩm

Thuật toán khai thác luật kết hợp xử lý dữ liệu chủ yếu bằng `product_id`. Tuy nhiên, để kết quả dễ hiểu trong báo cáo và ứng dụng demo, cần ánh xạ các mã sản phẩm này sang tên sản phẩm, quầy hàng và nhóm ngành hàng.

Cell này kiểm tra xem tất cả `product_id` trong `basket_matrix` có tồn tại trong `product_mapping_sample.csv` hay không. Nếu không có sản phẩm nào bị thiếu mapping, các luật kết hợp sau này có thể được hiển thị dưới dạng tên sản phẩm thay vì chỉ là mã số.

Ví dụ, thay vì hiển thị luật:

`24852 -> 21137`

hệ thống có thể hiển thị dưới dạng dễ hiểu hơn:

`Banana -> Organic Strawberries`

In [20]:
basket_product_ids = set([str(col) for col in basket_matrix.columns])
mapping_product_ids = set(product_mapping["product_id"].astype(str))

missing_in_mapping = basket_product_ids - mapping_product_ids
extra_in_mapping = mapping_product_ids - basket_product_ids

mapping_summary = pd.DataFrame([
    {
        "metric": "Số product_id trong basket_matrix",
        "value": len(basket_product_ids)
    },
    {
        "metric": "Số product_id trong product_mapping",
        "value": len(mapping_product_ids)
    },
    {
        "metric": "Số product_id trong matrix nhưng thiếu mapping",
        "value": len(missing_in_mapping)
    },
    {
        "metric": "Số product_id trong mapping nhưng không có trong matrix",
        "value": len(extra_in_mapping)
    },
    {
        "metric": "Số product_name duy nhất",
        "value": product_mapping["product_name"].nunique()
    },
    {
        "metric": "Số aisle_name duy nhất",
        "value": product_mapping["aisle_name"].nunique()
    },
    {
        "metric": "Số department_name duy nhất",
        "value": product_mapping["department_name"].nunique()
    }
])

display(mapping_summary)

,metric,value
0,Số product_id trong basket_matrix,3000
1,Số product_id trong product_mapping,3000
2,Số product_id trong matrix nhưng thiếu mapping,0
3,Số product_id trong mapping nhưng không có tro...,0
4,Số product_name duy nhất,2999
5,Số aisle_name duy nhất,113
6,Số department_name duy nhất,21


## 9. Lưu kết quả kiểm tra dữ liệu đầu vào

Sau khi hoàn tất quá trình kiểm tra, notebook lưu các bảng tổng hợp vào thư mục `results`. Các file này giúp ghi lại trạng thái dữ liệu đầu vào trước khi chuyển sang bước khai thác tập phổ biến.

Các file được tạo gồm:

| File kết quả | Nội dung |
| --- | --- |
| `input_file_check.csv` | Kiểm tra sự tồn tại và kích thước của các file đầu vào |
| `input_shape_summary.csv` | Tổng hợp số dòng, số cột và ý nghĩa của từng bảng dữ liệu |
| `input_mapping_summary.csv` | Kiểm tra ánh xạ giữa sản phẩm trong ma trận và bảng mapping |

Việc lưu các file kiểm tra này giúp quá trình làm bài có tính kiểm soát, dễ đối chiếu và dễ trình bày trong báo cáo.

In [21]:
file_check_df.to_csv(
    RESULTS_DIR / "input_file_check.csv",
    index=False,
    encoding="utf-8-sig"
)

shape_summary.to_csv(
    RESULTS_DIR / "input_shape_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

mapping_summary.to_csv(
    RESULTS_DIR / "input_mapping_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Trạng thái: Đã lưu kết quả kiểm tra input vào thư mục results.")

Trạng thái: Đã lưu kết quả kiểm tra input vào thư mục results.


## Tổng kết bước kiểm tra dữ liệu đầu vào

Sau bước kiểm tra dữ liệu đầu vào, có thể xác nhận rằng dữ liệu cuối kỳ đã được kế thừa đúng từ giai đoạn tiền xử lý giữa kỳ. Bộ dữ liệu đã sẵn sàng cho quá trình khai thác tập phổ biến và luật kết hợp.

Các điểm chính cần ghi nhận:

- Dữ liệu cuối kỳ bắt đầu từ các file đã xử lý trong `data/processed`, không xử lý lại từ dữ liệu Kaggle gốc.
- `basket_matrix_sparse.pkl` là đầu vào chính cho Apriori và FP-Growth.
- Ma trận giao dịch - sản phẩm có dạng nhị phân, phù hợp với bài toán khai thác luật kết hợp.
- `product_mapping_sample.csv` giúp chuyển kết quả từ mã sản phẩm sang tên sản phẩm dễ hiểu.
- `instacart_subset_for_rules.csv` sẽ được dùng cho các phân tích mở rộng như nhóm hàng, quầy hàng, thời gian mua hàng và hành vi mua lại.

Bước tiếp theo là sử dụng ma trận giao dịch - sản phẩm để khai thác frequent itemsets bằng Apriori và FP-Growth.